In [0]:
from pyspark.sql.functions import *

In [0]:
events_df = spark.read.table("live_nation_prod.bronze_prod.events")
sales_df = spark.read.table("live_nation_prod.bronze_prod.ticket_sales")

In [0]:
events_df.columns

In [0]:
# From my Exploratort Notebook, I found out that there are different formats for the concert and a lot of add ones. This change cleans the name of the events.

events_df = events_df.withColumn(
    "event_name",
    when(
        col("event_name") == "New Harbor: Super Awesome Tour",
        "Neon Harbor - Super Awesome Tour"
    ).otherwise(col("event_name"))
)

In [0]:
# This statement creates a new field that categorizes a record as an event or an upsell. 
events_df = events_df.withColumn(
    "event_type",
    when(
        col("event_name") == 'Neon Harbor - Super Awesome Tour','event'
    ).otherwise('upsell')
)

In [0]:
display(events_df.limit(100))

In [0]:
# I created a temp view of the cleaned events table. This is will make the SQL statement cleaner and more readable.
events_df.createOrReplaceTempView("events_silver")

In [0]:
%sql
-- Save the temp view to the the silver layer schema. This keeps the source tables intact while creating a cleaner dataset to query in the future.
CREATE OR REPLACE TABLE live_nation_prod.silver_prod.events_silver AS SELECT * FROM events_silver
 

**Question 1 Query**

In [0]:
%sql
-- I kept the the event type fields as a way of verifying the results. 
-- Creating the Temp table first made this query much more straight forward. 
SELECT 
    e.event_id AS event_event_id,
    e.event_name AS event_event_name,
    e.venue_id AS event_venue_id,
    e.event_dt AS event_event_dt,
    e.event_type AS event_event_type,
    u.event_id AS upsell_event_id,
    u.event_name AS upsell_event_name,
    u.event_id AS upsell_venue_id,
    u.event_dt AS upsell_event_dt,
    u.event_type AS upsell_event_type

FROM events_silver e
INNER JOIN events_silver u
    ON e.venue_id = u.venue_id
WHERE e.event_type = 'event'
  AND u.event_type = 'upsell'
